In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [1]:
!pip install -q -U ultralytics
import ultralytics
print("Ultralytics version:", ultralytics.__version__)
!nvidia-smi

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.7/45.7 kB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 25.8 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.2/64.2 kB 3.9 MB/s eta 0:00:00
Creating new Ultralytics Settings v0.0.7 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart#ultralytics-settings.
Ultralytics version: 8.4.126
Sat Aug 22 10:00:24 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.159.04             Driver Version: 580.159.04     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Per

In [3]:
import os
for root, dirs, files in os.walk('/kaggle/input'):
    level = root.replace('/kaggle/input', '').count(os.sep)
    indent = '  ' * level
    print(f'{indent}{os.path.basename(root)}/')
    if level < 2:
        for f in files[:5]:
            print(f'{indent}  {f}')

input/
  datasets/
    sayedgamal99/
      smoke-fire-detection-yolo/
        data/
          val/
            labels/
            images/
          test/
            labels/
            images/
          train/
            labels/
            images/
    alinoorqureshi/
      weapon-detection-yolo-optimized/
        dataset_merged/
          val/
            labels/
            images/
          test/
            labels/
            images/
          train/
            labels/
            images/


In [4]:
DATA_YAML_ORIGINAL = "/kaggle/input/datasets/sayedgamal99/smoke-fire-detection-yolo/data.yaml"

with open(DATA_YAML_ORIGINAL) as f:
    print(f.read())

path: /kaggle/working/D Fire Dataset  # dataset root dir
train: data/train/images  # train images (relative to 'path')
val: data/val/images  # val images (relative to 'path')
test: data/test/images  # test images (relative to 'path')

# Classes
names: ['smoke', 'fire']  # Replace with your actual class names

# Counts
nc: 2  # number of classes
train_count: 14122
val_count: 3099
test_count: 4306



In [5]:
BASE = "/kaggle/input/datasets/sayedgamal99/smoke-fire-detection-yolo/data"

corrected_yaml = f"""
train: {BASE}/train/images
val: {BASE}/val/images
test: {BASE}/test/images
nc: 2
names: ['smoke', 'fire']
"""

with open("/kaggle/working/data.yaml", "w") as f:
    f.write(corrected_yaml)

DATA_YAML = "/kaggle/working/data.yaml"
print("Using corrected data.yaml:")
with open(DATA_YAML) as f:
    print(f.read())

Using corrected data.yaml:

train: /kaggle/input/datasets/sayedgamal99/smoke-fire-detection-yolo/data/train/images
val: /kaggle/input/datasets/sayedgamal99/smoke-fire-detection-yolo/data/val/images
test: /kaggle/input/datasets/sayedgamal99/smoke-fire-detection-yolo/data/test/images
nc: 2
names: ['smoke', 'fire']



In [6]:
from ultralytics import YOLO

yolo11_model = YOLO("yolo11n.pt")

yolo11_results = yolo11_model.train(
    data=DATA_YAML,
    epochs=15,
    imgsz=640,
    batch=32,          # bumped up since we're using 2 GPUs
    device=[0, 1],      # use both T4s
    project="/kaggle/working/runs",
    name="yolo11n_finetuned",
)

Ultralytics 8.4.126 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
                                                        CUDA:1 (Tesla T4, 14912MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=False, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/working/data.yaml, degrees=0.0, deterministic=True, device=0,1, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=15, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11n.pt,

In [7]:
from ultralytics import YOLO

yolo11_model = YOLO("yolo11n.pt")

yolo11_results = yolo11_model.train(
    data=DATA_YAML,
    epochs=15,
    imgsz=640,
    batch=32,          # bumped up since we're using 2 GPUs
    device=[0, 1],      # use both T4s
    project="/kaggle/working/runs",
    name="yolo11n_finetuned",
)

Ultralytics 8.4.126 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
                                                        CUDA:1 (Tesla T4, 14912MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=False, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/working/data.yaml, degrees=0.0, deterministic=True, device=0,1, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=15, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11n.pt,

In [8]:
from ultralytics import RTDETR

rtdetr_model = RTDETR("rtdetr-l.pt")

rtdetr_results = rtdetr_model.train(
    data=DATA_YAML,
    epochs=15,
    imgsz=640,
    batch=16,           # RT-DETR is heavier per-image than YOLO, kept lower than YOLO's batch
    device=[0, 1],
    project="/kaggle/working/runs",
    name="rtdetr_l_finetuned",
)

Ultralytics 8.4.126 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
                                                        CUDA:1 (Tesla T4, 14912MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=False, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/working/data.yaml, degrees=0.0, deterministic=True, device=0,1, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=15, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=rtdetr-l.pt

/usr/local/lib/python3.12/dist-packages/torch/autograd/graph.py:865: UserWarning: grid_sampler_2d_backward_cuda does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:160.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass
/usr/local/lib/python3.12/dist-packages/torch/autograd/graph.py:865: UserWarning: grid_sampler_2d_backward_cuda does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:160.)
  return Variable._execution

       1/15      6.73G     0.7174      1.587     0.4371          9        640: 100% ━━━━━━━━━━━━ 882/882 1.5it/s 9:520.9ss
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 97/97 2.9it/s 33.2s0.3ss
                   all       3094       3917      0.494      0.392       0.37      0.163

      Epoch    GPU_mem  giou_loss   cls_loss    l1_loss  Instances       Size


/usr/local/lib/python3.12/dist-packages/torch/autograd/graph.py:865: UserWarning: grid_sampler_2d_backward_cuda does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:160.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass
/usr/local/lib/python3.12/dist-packages/torch/autograd/graph.py:865: UserWarning: grid_sampler_2d_backward_cuda does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:160.)
  return Variable._execution

       2/15      7.18G      0.646     0.8902     0.3596          6        640: 100% ━━━━━━━━━━━━ 882/882 1.6it/s 9:250.6ss
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 97/97 3.1it/s 31.0s0.3ss
                   all       3094       3917      0.387      0.368      0.317      0.137

      Epoch    GPU_mem  giou_loss   cls_loss    l1_loss  Instances       Size


...

      13/15      7.18G     0.5185     0.7369     0.3144          1        640: 100% ━━━━━━━━━━━━ 882/882 1.6it/s 9:090.6ss
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 97/97 3.1it/s 31.2s0.3ss
                   all       3094       3917      0.689      0.614       0.66      0.362

      Epoch    GPU_mem  giou_loss   cls_loss    l1_loss  Instances       Size


/usr/local/lib/python3.12/dist-packages/torch/autograd/graph.py:865: UserWarning: grid_sampler_2d_backward_cuda does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:160.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass
/usr/local/lib/python3.12/dist-packages/torch/autograd/graph.py:865: UserWarning: grid_sampler_2d_backward_cuda does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:160.)
  return Variable._execution

      14/15      7.18G     0.5162      0.691     0.3186          2        640: 100% ━━━━━━━━━━━━ 882/882 1.6it/s 9:080.5ss
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 97/97 3.1it/s 31.0s0.3ss
                   all       3094       3917      0.692      0.642      0.684      0.374

      Epoch    GPU_mem  giou_loss   cls_loss    l1_loss  Instances       Size


/usr/local/lib/python3.12/dist-packages/torch/autograd/graph.py:865: UserWarning: grid_sampler_2d_backward_cuda does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:160.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass
/usr/local/lib/python3.12/dist-packages/torch/autograd/graph.py:865: UserWarning: grid_sampler_2d_backward_cuda does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:160.)
  return Variable._execution

      15/15      7.18G     0.5049     0.6475      0.309          3        640: 100% ━━━━━━━━━━━━ 882/882 1.6it/s 9:110.5ss
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 97/97 3.1it/s 31.2s0.3ss
                   all       3094       3917      0.707      0.656      0.706      0.393

15 epochs completed in 2.464 hours.
Optimizer stripped from /kaggle/working/runs/rtdetr_l_finetuned/weights/last.pt, 66.2MB
Optimizer stripped from /kaggle/working/runs/rtdetr_l_finetuned/weights/best.pt, 66.2MB

Validating /kaggle/working/runs/rtdetr_l_finetuned/weights/best.pt...
rt-detr-l summary: 315 layers, 31,987,850 parameters, 0 gradients, 105.3 GFLOPs
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 97/97 3.7it/s 26.5s0.3ss
                   all       3094       3917      0.708      0.656      0.708      0.394
                 smoke       1545       1751      0.735       0.68   

In [9]:
rtdetr_best = "/kaggle/working/runs/rtdetr_l_finetuned/weights/best.pt"
rtdetr_val_model = RTDETR(rtdetr_best)
rtdetr_metrics = rtdetr_val_model.val(data=DATA_YAML)

print("RT-DETR-L — mAP50-95:", rtdetr_metrics.box.map)
print("RT-DETR-L — mAP50:", rtdetr_metrics.box.map50)

Ultralytics 8.4.126 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
rt-detr-l summary: 315 layers, 31,987,850 parameters, 0 gradients, 105.3 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 157.1±203.5 MB/s, size: 122.4 KB)
val: Scanning /kaggle/input/datasets/sayedgamal99/smoke-fire-detection-yolo/data/val/labels... 3094 images, 1375 backgrounds, 5 corrupt: 100% ━━━━━━━━━━━━ 3099/3099 1.1Kit/s 2.8s<0.1s
val: /kaggle/input/datasets/sayedgamal99/smoke-fire-detection-yolo/data/val/images/WEB07535.jpg: ignoring corrupt image/label: [Errno 30] Read-only file system: '/kaggle/input/datasets/sayedgamal99/smoke-fire-detection-yolo/data/val/images/WEB07535.jpg'
val: /kaggle/input/datasets/sayedgamal99/smoke-fire-detection-yolo/data/val/images/WEB07536.jpg: ignoring corrupt image/label: [Errno 30] Read-only file system: '/kaggle/input/datasets/sayedgamal99/smoke-fire-detection-yolo/data/val/images/WEB07536.jpg'
val: /kaggle/input/datasets/sayedgamal99/smoke-fire-detectio

In [11]:
from ultralytics import YOLO

yolo11_best = "/kaggle/working/runs/yolo11n_finetuned/weights/best.pt"

yolo11_val_model = YOLO(yolo11_best)

yolo11_metrics = yolo11_val_model.val(data=DATA_YAML)

print("YOLOv11n — mAP50-95:", yolo11_metrics.box.map)
print("YOLOv11n — mAP50:", yolo11_metrics.box.map50)

Ultralytics 8.4.126 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
YOLO11n summary (fused): 101 layers, 2,582,542 parameters, 0 gradients, 6.4 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 110.1±91.6 MB/s, size: 47.9 KB)
val: Scanning /kaggle/input/datasets/sayedgamal99/smoke-fire-detection-yolo/data/val/labels... 3094 images, 1375 backgrounds, 5 corrupt: 100% ━━━━━━━━━━━━ 3099/3099 1.1Kit/s 2.8s0.0ss
val: /kaggle/input/datasets/sayedgamal99/smoke-fire-detection-yolo/data/val/images/WEB07535.jpg: ignoring corrupt image/label: [Errno 30] Read-only file system: '/kaggle/input/datasets/sayedgamal99/smoke-fire-detection-yolo/data/val/images/WEB07535.jpg'
val: /kaggle/input/datasets/sayedgamal99/smoke-fire-detection-yolo/data/val/images/WEB07536.jpg: ignoring corrupt image/label: [Errno 30] Read-only file system: '/kaggle/input/datasets/sayedgamal99/smoke-fire-detection-yolo/data/val/images/WEB07536.jpg'
val: /kaggle/input/datasets/sayedgamal99/smoke-fire-detecti

In [12]:
from ultralytics import RTDETR

rtdetr_best = "/kaggle/working/runs/rtdetr_l_finetuned/weights/best.pt"

rtdetr_val_model = RTDETR(rtdetr_best)

rtdetr_metrics = rtdetr_val_model.val(data=DATA_YAML)

print("RT-DETR-L — mAP50-95:", rtdetr_metrics.box.map)
print("RT-DETR-L — mAP50:", rtdetr_metrics.box.map50)

Ultralytics 8.4.126 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
rt-detr-l summary: 315 layers, 31,987,850 parameters, 0 gradients, 105.3 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 330.4±194.7 MB/s, size: 179.0 KB)
val: Scanning /kaggle/input/datasets/sayedgamal99/smoke-fire-detection-yolo/data/val/labels... 3094 images, 1375 backgrounds, 5 corrupt: 100% ━━━━━━━━━━━━ 3099/3099 1.1Kit/s 2.8s<0.0s
val: /kaggle/input/datasets/sayedgamal99/smoke-fire-detection-yolo/data/val/images/WEB07535.jpg: ignoring corrupt image/label: [Errno 30] Read-only file system: '/kaggle/input/datasets/sayedgamal99/smoke-fire-detection-yolo/data/val/images/WEB07535.jpg'
val: /kaggle/input/datasets/sayedgamal99/smoke-fire-detection-yolo/data/val/images/WEB07536.jpg: ignoring corrupt image/label: [Errno 30] Read-only file system: '/kaggle/input/datasets/sayedgamal99/smoke-fire-detection-yolo/data/val/images/WEB07536.jpg'
val: /kaggle/input/datasets/sayedgamal99/smoke-fire-detectio

In [13]:
comparison_md = f"""# Model Comparison — YOLOv11 vs RT-DETR-L

| Model | mAP50-95 | mAP50 |
|---|---|---|
| YOLOv11n | {yolo11_metrics.box.map:.4f} | {yolo11_metrics.box.map50:.4f} |
| RT-DETR-L | {rtdetr_metrics.box.map:.4f} | {rtdetr_metrics.box.map50:.4f} |
"""

with open("/kaggle/working/MODEL_COMPARISON_KAGGLE.md", "w") as f:
    f.write(comparison_md)

print(comparison_md)

# Model Comparison — YOLOv11 vs RT-DETR-L

| Model | mAP50-95 | mAP50 |
|---|---|---|
| YOLOv11n | 0.3976 | 0.7073 |
| RT-DETR-L | 0.3931 | 0.7059 |



In [14]:
from ultralytics import YOLO

yolov8_model = YOLO("yolov8n.pt")

yolov8_results = yolov8_model.train(
    data=DATA_YAML,
    epochs=15,
    imgsz=640,
    batch=32,
    device=[0, 1],
    project="/kaggle/working/runs",
    name="yolov8n_finetuned"
)

Ultralytics 8.4.126 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
                                                        CUDA:1 (Tesla T4, 14912MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=False, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/working/data.yaml, degrees=0.0, deterministic=True, device=0,1, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=15, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt,

In [16]:
from ultralytics import YOLO

yolo26_model = YOLO("yolo26n.pt")

yolo26_results = yolo26_model.train(
    data=DATA_YAML,
    epochs=15,
    imgsz=640,
    batch=32,
    device=[0, 1],
    project="/kaggle/working/runs",
    name="yolo26n_finetuned"
)

Ultralytics 8.4.126 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
                                                        CUDA:1 (Tesla T4, 14912MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=False, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/working/data.yaml, degrees=0.0, deterministic=True, device=0,1, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=15, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo26n.pt,

In [19]:
from ultralytics import YOLO

yolov8_best = "/kaggle/working/runs/yolov8n_finetuned/weights/best.pt"

yolov8_val_model = YOLO(yolov8_best)

yolov8_metrics = yolov8_val_model.val(data=DATA_YAML)

print("YOLOv8n mAP50-95:", yolov8_metrics.box.map)
print("YOLOv8n mAP50:", yolov8_metrics.box.map50)

Ultralytics 8.4.126 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
Model summary (fused): 73 layers, 3,006,038 parameters, 0 gradients, 8.1 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 254.5±249.9 MB/s, size: 129.4 KB)
val: Scanning /kaggle/input/datasets/sayedgamal99/smoke-fire-detection-yolo/data/val/labels... 3094 images, 1375 backgrounds, 5 corrupt: 100% ━━━━━━━━━━━━ 3099/3099 1.1Kit/s 2.8s<0.1s
val: /kaggle/input/datasets/sayedgamal99/smoke-fire-detection-yolo/data/val/images/WEB07535.jpg: ignoring corrupt image/label: [Errno 30] Read-only file system: '/kaggle/input/datasets/sayedgamal99/smoke-fire-detection-yolo/data/val/images/WEB07535.jpg'
val: /kaggle/input/datasets/sayedgamal99/smoke-fire-detection-yolo/data/val/images/WEB07536.jpg: ignoring corrupt image/label: [Errno 30] Read-only file system: '/kaggle/input/datasets/sayedgamal99/smoke-fire-detection-yolo/data/val/images/WEB07536.jpg'
val: /kaggle/input/datasets/sayedgamal99/smoke-fire-detectio

In [20]:
from ultralytics import YOLO

yolo26_best = "/kaggle/working/runs/yolo26n_finetuned/weights/best.pt"

yolo26_val_model = YOLO(yolo26_best)

yolo26_metrics = yolo26_val_model.val(data=DATA_YAML)

print("YOLO26n mAP50-95:", yolo26_metrics.box.map)
print("YOLO26n mAP50:", yolo26_metrics.box.map50)

Ultralytics 8.4.126 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
YOLO26n summary (fused): 122 layers, 2,375,226 parameters, 0 gradients, 5.3 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 309.6±148.7 MB/s, size: 164.8 KB)
val: Scanning /kaggle/input/datasets/sayedgamal99/smoke-fire-detection-yolo/data/val/labels... 3094 images, 1375 backgrounds, 5 corrupt: 100% ━━━━━━━━━━━━ 3099/3099 1.0Kit/s 3.0s<0.0s
val: /kaggle/input/datasets/sayedgamal99/smoke-fire-detection-yolo/data/val/images/WEB07535.jpg: ignoring corrupt image/label: [Errno 30] Read-only file system: '/kaggle/input/datasets/sayedgamal99/smoke-fire-detection-yolo/data/val/images/WEB07535.jpg'
val: /kaggle/input/datasets/sayedgamal99/smoke-fire-detection-yolo/data/val/images/WEB07536.jpg: ignoring corrupt image/label: [Errno 30] Read-only file system: '/kaggle/input/datasets/sayedgamal99/smoke-fire-detection-yolo/data/val/images/WEB07536.jpg'
val: /kaggle/input/datasets/sayedgamal99/smoke-fire-detec

In [21]:
comparison_md = f"""# Model Comparison

| Model | mAP50-95 | mAP50 |
|---|---|---|
| YOLOv8n | {yolov8_metrics.box.map:.4f} | {yolov8_metrics.box.map50:.4f} |
| YOLOv11n | {yolo11_metrics.box.map:.4f} | {yolo11_metrics.box.map50:.4f} |
| YOLO26n | {yolo26_metrics.box.map:.4f} | {yolo26_metrics.box.map50:.4f} |
| RT-DETR-L | {rtdetr_metrics.box.map:.4f} | {rtdetr_metrics.box.map50:.4f} |
"""

print(comparison_md)

# Model Comparison

| Model | mAP50-95 | mAP50 |
|---|---|---|
| YOLOv8n | 0.4017 | 0.7089 |
| YOLOv11n | 0.3976 | 0.7073 |
| YOLO26n | 0.3813 | 0.6804 |
| RT-DETR-L | 0.3931 | 0.7059 |

